# Suficiencia de la evidencia

**Fase:** E15

## Objetivo

Calibrar, solo sobre validación, el criterio que decide si la evidencia disponible sostiene una explicación.

## Relación con la pregunta de investigación

> ¿Puede un agente con RAG explicar el diagnóstico de un vehículo a partir de datos OBD-II, rastreando cada explicación hasta su evidencia de origen, y reconocer cuándo la evidencia disponible no es suficiente para sostener una explicación en lugar de generar una hipótesis sin fundamento?

Es el componente que la PI interroga directamente: reconocer cuándo la evidencia no basta.

## Hipótesis

_Por declarar antes de ejecutar la fase._

## Datos

_Dataset, versión y partición utilizados._

## Método

_Procedimiento, parámetros y semillas._

## Métricas

_Qué se mide y con qué instrumento._

## Criterio de éxito o refutación

_Fijado antes de mirar los resultados._


In [1]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))

import pandas as pd

from mecanico_ia_exp.config import paths


## El criterio y sus señales

Conjunción, no promedio: ningún factor es compensable.

In [2]:
import json
s = json.loads((paths.EXPERIMENTS / 'E15_suficiencia.json').read_text())
print('umbrales:', json.dumps(s['thresholds'], indent=2, ensure_ascii=False))
print('calibración:', s['calibration'])

umbrales: {
  "min_confidence": 0.6,
  "min_retrieval_score": 0.35,
  "min_supporting_chunks": 2,
  "min_distinct_sources": 1,
  "min_claim_coverage": 0.8,
  "allow_compatible_applicability": false,
  "accepted_applicability": [
    "EXACT_MATCH"
  ]
}
calibración: isotonic, cv=5 sobre entrenamiento


In [3]:
filas = [{'caso': c['case_id'], 'decisión': c['verdict']['decision'],
          **{k: v for k, v in c['verdict']['signals'].items()
             if k in ('confidence', 'n_documents_applicable',
                      'n_distinct_sources', 'claim_coverage')},
          'motivos': ', '.join(c['verdict']['reasons'])}
         for c in s['cases']]
pd.DataFrame(filas)

,caso,decisión,confidence,n_documents_applicable,n_distinct_sources,claim_coverage,motivos
0,C01_maf_drift,ABSTAIN,0.5999,7,1,0.0,"LOW_MODEL_CONFIDENCE, LOW_RETRIEVAL_SUPPORT, L..."
1,C02_thermostat_stuck,EXPLAIN,1.0000,8,1,1.0,
2,C03_o2_slow_response,EXPLAIN,1.0000,6,1,1.0,
3,C04_misfire,ABSTAIN,0.4621,4,1,1.0,"LOW_MODEL_CONFIDENCE, LOW_RETRIEVAL_SUPPORT"
4,C05_vacuum_leak,EXPLAIN,0.9078,4,1,1.0,
5,C06_vacuum_leak_evidencia_retirada,ABSTAIN,0.9078,0,0,0.0,"NO_APPLICABLE_DOCUMENTARY_EVIDENCE, LOW_CLAIM_..."


### Citar no es sostener

El verificador rechaza las citas que no hablan de la señal afirmada.

In [4]:
checks = [ch for c in s['cases'] for ch in c['support_checks']]
print(f'comprobaciones: {len(checks)} · aceptadas: '
      f'{sum(1 for x in checks if x["supports"])}')
pd.Series([x['reason'] for x in checks if not x['supports']]).value_counts()

comprobaciones: 51 · aceptadas: 21


el fragmento no menciona la señal stft_bank1       14
el fragmento no menciona la señal oxygen_sensor     9
el fragmento no menciona la señal ltft_bank1        7
Name: count, dtype: int64

## Resultados

_Lo observado, sin interpretación._

## Interpretación

_Qué significa, y qué no significa._

## Limitaciones

_Qué queda fuera del alcance de lo mostrado._

## Conclusión experimental

_Respuesta al criterio fijado en la cabecera._

## Artefactos generados

_Ficheros producidos, con su ruta._
